### Checks

`Urban Ward_Panel` sheets.


1. All TV_CODES should show up at least once. Either one shape for a town or village (NaN Ward code) or multiple ward shapes with codes.
2. Which TV_Codes have we been given multiple shapes for (wards)? Should match or overdeliver on their promise.

Add column that says "Delivery State": 
- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd

gpd.options.io_engine = "pyogrio"

In [ ]:
from gridsample.utils import save_shapefiles

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data_panel"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data"
CLEANED_DATA_DIR = (
    DATA_DIR / "02. Intermediate Outputs" / "v1" / "00. Merge and Quality Checks"
)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 0. Load request excel

In [ ]:
excel_file = (
    RAW_DATA_DIR / "00. Boundary Requests" / "SamplingOutput_Summary_IFS & Panel.xlsx"
)

In [ ]:
panel_df = pd.read_excel(excel_file, sheet_name="Urban Ward_Panel")

In [ ]:
panel_df

In [ ]:
# sort and reset index
panel_df = panel_df.sort_values(
    by=["State", "District", "Subdistrict", "TownVillage", "UrbanWardVillage"]
).reset_index(drop=True)

In [ ]:
panel_df

In [ ]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards.csv", index=False)

In [ ]:
# get unique across district and subdistrict both
len(panel_df[["State", "District", "Subdistrict"]].drop_duplicates())

#### Check for duplicated wards in our own sample

In [ ]:
duplicated_sampled_wards = panel_df[
    panel_df[["TownVillage", "UrbanWardVillage"]].duplicated(keep=False)
].sort_values(["TownVillage", "UrbanWardVillage"])
duplicated_sampled_wards

In [ ]:
duplicated_sampled_wards.to_csv(
    CLEANED_DATA_DIR / "Duplicated Sampled Wards.csv", index=False
)

## 1. Load all boundaries

In [ ]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]

In [ ]:
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True)

In [ ]:
gdf = gdf.drop_duplicates()

#### Fix Chittoor issue

In [ ]:
# gdf[gdf["Dist_N"] == "Chittoor"]

In [ ]:
gdf.loc[gdf["TV_C"] == 803014, ["SubDist_N", "SubDist_C"]] = ["Tirupati (Urban)", 5383]

In [ ]:
# gdf[gdf["Dist_N"] == "Chittoor"]

#### Fix Patna ward PCA_ID issue

In [ ]:
# gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3")]

In [ ]:
gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3"), "PCA_ID"] = "801373-3"

In [ ]:
# gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3")]

#### Replace some MDDS codes with TV and Ward codes as census expects

In [ ]:
# gdf.loc[(gdf["TV_C"].isin([519923, 574077]))]

In [ ]:
gdf.loc[gdf["TV_C"] == 519923, ["TV_C", "Ward_C", "PCA_ID"]] = [802596, 18, "802596-18"]
gdf.loc[gdf["TV_C"] == 574077, ["TV_C", "Ward_C", "PCA_ID"]] = [802918, 157, "802918-157"]

## 2. Checks

### Any MapSolve rows with missing TV code?

In [ ]:
gdf_no_TV_code_filtered = gdf[gdf["TV_C"].isna()]
gdf_no_TV_code_filtered

In [ ]:
gdf_no_TV_code_filtered.to_csv(
    CLEANED_DATA_DIR / "MapSolve Data without TV Codes.csv", index=False
)

### Request satisfaction check

In [ ]:
given_states_list = list(gdf["State_C"].unique())
given_states_list.append(
    90
)  # manually add 90 for Telangana / Andhra Pradesh discrepency
len(given_states_list)

In [ ]:
panel_df.loc[panel_df["State"].isin(given_states_list), "State_Name"].unique()

In [ ]:
panel_df["State Shared by MapSolve"] = False
panel_df.loc[
    panel_df["State"].isin(given_states_list), "State Shared by MapSolve"
] = True

In [ ]:
len(panel_df[~panel_df["State Shared by MapSolve"]])

In [ ]:
panel_df["State Changed"] = "No"
panel_df.loc[panel_df["State_Name"] == "TELANGANA", "State Changed"] = (
    "Previously Andhra Pradesh"
)

In [ ]:
panel_df

#### Check for wards

In [ ]:
panel_df["PCA_ID"] = (
    panel_df["TownVillage"].astype(str)
    + "-"
    + panel_df["UrbanWardVillage"].astype(str)
)
given_ward_ids = gdf["PCA_ID"].unique()

panel_df["Ward Boundary Given"] = False
panel_df.loc[panel_df["PCA_ID"].isin(given_ward_ids), "Ward Boundary Given"] = True

In [ ]:
panel_df["PCA_ID"]

In [ ]:
len(set(panel_df["PCA_ID"]).intersection(given_ward_ids))

#### Check for TownVillage codes

In [ ]:
given_TV_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].notna(),
        "TV_C",
    ].astype(int)
)

panel_df["TV Boundary Given"] = False
panel_df.loc[
    panel_df["TownVillage"].astype(int).isin(given_TV_ids),
    "TV Boundary Given",
] = True

#### Check for SubDistricts

In [ ]:
given_subdist_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].isna() & gdf["SubDist_C"].notna(),
        "SubDist_C",
    ].astype(int)
)

panel_df["SubDistrict Boundary Given"] = False
panel_df.loc[
    panel_df["Subdistrict"].astype(int).isin(given_subdist_ids),
    "SubDistrict Boundary Given",
] = True

#### Fill in Delivery State

- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

In [ ]:
## baseline
panel_df["Delivery State"] = "BAD - No boundary(s) given"

## subdistrict
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "BAD - Subdistrict boundary given but Ward boundaries expected"

# tripura > dukli special case
panel_df.loc[
    (panel_df["State_Name"] == "TRIPURA") & (panel_df["Subd_Name"] == "Dukli"),
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"

## town/village
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "GOOD - Town/Village boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "BAD - Town/Village boundary given but Ward boundaries expected"

## ward
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "BETTER - Ward boundary given but only TV or Subdistrict was expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "GOOD - Ward boundary given as expected"

In [ ]:
panel_df["Delivery State"].value_counts()

#### Add `PSU Type` column

In [ ]:
psu_mapping = {
    "BETTER - Ward boundary given but only TV or Subdistrict was expected": "ward",
    "GOOD - Town/Village boundary given as expected": "town_village",
    "GOOD - Ward boundary given as expected": "ward",
    "GOOD - Subdistrict boundary given as expected": "subdistrict",
    "BAD - Town/Village boundary given but Ward boundaries expected": "town_village",
    "BAD - No boundary(s) given": "none",
    "BAD - Subdistrict boundary given but Ward boundaries expected": "subdistrict",
}
# create a new column in panel_df called PSU Type
panel_df["PSU Type"] = panel_df["Delivery State"].map(psu_mapping)

#### Add `PSU ID` column (unique overall ID)

In [ ]:
# add an ID column that is unique across all rows. It should be WARD_{PCA_ID} if the PSU Type is "ward", TV_{TV Code} if the PSU Type is "town_village", and SUBDISTRICT_{Subdistrict Code} if the PSU Type is "subdistrict"
panel_df["PSU ID"] = panel_df.apply(
    lambda row: f"WARD_{row['PCA_ID']}"
    if row["PSU Type"] == "ward"
    else f"TV_{int(row['TownVillage'])}"
    if row["PSU Type"] == "town_village"
    else f"SUBDISTRICT_{int(row['Subdistrict'])}",
    axis=1,
).astype(str)

#### Add `Ward Count` column

In [ ]:
panel_df["Ward Count"] = 0

panel_df.loc[panel_df["PSU Type"] == "ward", "Ward Count"] = 1
panel_df.loc[panel_df["PSU Type"] == "town_village", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "town_village"]
    .groupby("TownVillage")["UrbanWardVillage"]
    .transform("count")
)
panel_df.loc[panel_df["PSU Type"] == "subdistrict", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "subdistrict"]
    .groupby("Subdistrict")["UrbanWardVillage"]
    .transform("count")
)
panel_df["Ward Count"] = panel_df["Ward Count"].astype(int)

#### Reorder columns

In [ ]:
reordered_columns = [
    "State",
    "State_Name",
    "District",
    "District_Name",
    "Subdistrict",
    "Subd_Name",
    "TownVillage",
    "UrbanWardVillage",
    "WardVillage_Name",
    "PCA_ID",
    "TRU",
    "WardVillage_Pop",
    "Subd_Pop",
    "State_Pop",
    "WardVillageID",
    "Ward Boundary Available with MapSolve",
    "Included in Panel",
    "State Shared by MapSolve",
    "State Changed",
    "Ward Boundary Given",
    "TV Boundary Given",
    "SubDistrict Boundary Given",
    "Delivery State",
    "PSU Type",
    "PSU ID",
    "Ward Count",
]
panel_df = panel_df[reordered_columns]

#### Save

In [ ]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards with Quality Checks.csv", index=False)

In [ ]:
if not panel_df["State Shared by MapSolve"].all():
    print("There are states in the merged data that are not shared by MapSolve:")
    print(panel_df[~panel_df["State Shared by MapSolve"]]["State"].unique())
    print("Saving the states for which we DO have data separately:")
    panel_df[panel_df["State Shared by MapSolve"]].to_csv(
        CLEANED_DATA_DIR / "Panel Wards with Quality Checks - Shared States.csv",
        index=False,
    )

#### Save per-state stats

In [ ]:
# Get counts of delivery states by state
delivery_state_counts = (
    panel_df[panel_df["State"].isin(given_states_list)]
    .groupby("State")["Delivery State"]
    .value_counts()
)

# Convert to a more readable DataFrame format
delivery_state_pivot = delivery_state_counts.unstack(fill_value=0).reset_index()

# Sort by state code
delivery_state_pivot = delivery_state_pivot.sort_values(by="State")

# Add state names for better readability
state_name_mapping = (
    panel_df[["State", "State_Name"]]
    .drop_duplicates()
    .set_index("State")["State_Name"]
)
delivery_state_pivot["State_Name"] = delivery_state_pivot["State"].map(
    state_name_mapping
)

# Reorder columns to show State_Name first
columns = ["State", "State_Name"] + [
    col for col in delivery_state_pivot.columns if col not in ["State", "State_Name"]
]
delivery_state_pivot = delivery_state_pivot[columns]

# transform the DataFrame to have the delivery states as rows and state names as columns
delivery_state_pivot.drop(columns=["State"], inplace=True)
delivery_state_pivot = delivery_state_pivot.set_index("State_Name").T.reset_index()
delivery_state_pivot

In [ ]:
# add a total column as the second column
delivery_state_pivot.insert(1, "Total", delivery_state_pivot.iloc[:, 1:].sum(axis=1))

In [ ]:
delivery_state_pivot

In [ ]:
delivery_state_pivot.to_csv(
    CLEANED_DATA_DIR / "Delivery State Counts By State.csv", index=False
)